# 01 — Smoke Test

Quick end-to-end check that the scaffolding wires up correctly.

Runs against the stub models (no GPU required) — swap in real weights before serious testing.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import yaml
with open('../configs/default.yaml') as f:
    cfg = yaml.safe_load(f)
print(cfg.keys())

## 1. Glossary lookup

In [ ]:
from src.context.glossary import Glossary
g = Glossary.from_json('../data/glossaries/ml-conference-en-ko.json')
hits = g.hits_for('We fine-tuned a large language model with PyTorch for inference.')
for h in hits:
    print(f'  {h.src:<20} → {h.tgt}  (dnt={h.dnt})')

## 2. Translation memory retrieval

In [ ]:
from src.context.translation_memory import TranslationMemory
tm = TranslationMemory(cfg)
tm.load_jsonl('../data/translation_memory/en-ko.jsonl')
for r in tm.retrieve('The model achieved strong results.')[:3]:
    print(f'  [{r.score:.2f}] {r.src}')
    print(f'         → {r.tgt}\n')

## 3. Full prompt build

In [ ]:
from src.context.prompt_builder import PromptBuilder, SessionContext
pb = PromptBuilder(cfg)
ctx = SessionContext(
    src_lang='en', tgt_lang='ko',
    topic_summary='Machine learning conference talk',
    speaker_name='Alice', register='formal',
)
ctx.prev_src_segments = ['Good morning, everyone.']
ctx.prev_tgt_segments = ['안녕하세요, 여러분.']

src = 'We fine-tuned an LLM using PyTorch for faster inference.'
prompt = pb.build_system_prompt(ctx, g.hits_for(src), tm.retrieve(src))
print(prompt)

## 4. Full pipeline (stub models)

In [ ]:
import asyncio, numpy as np
from src.pipeline.session import SessionConfig, TranslationSession
from src.pipeline.streaming_loop import run_streaming_loop

session_cfg = SessionConfig(src_lang='en', tgt_lang='ko', speaker_id='demo_speaker')
session = TranslationSession(cfg, session_cfg)

async def smoke():
    audio_q = asyncio.Queue()
    out_q = asyncio.Queue()
    chunk = np.zeros(int(2.0 * 16000), dtype=np.float32)
    for _ in range(3):
        await audio_q.put(chunk)
    await audio_q.put(None)
    await run_streaming_loop(session, audio_q, out_q)
    events = []
    while not out_q.empty():
        e = out_q.get_nowait()
        if e is not None: events.append(e)
    return events

events = await smoke()
print(f'Events: {len(events)}')
for e in events:
    print(f'  {e.latency_ms:.0f}ms | {e.src_delta!r} → {e.tgt_delta!r}')
session.close()